# Stage 2: Exploratory Data Analysis (EDA)
This notebook visualizes Indian inflation dynamics from 2013 to 2025. It creates the year-month heatmap of core inflation, plots the component trends against the RBI repo rate, and computes variance decomposition (correlations and annual trends).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

## 1. Load Data

In [ ]:
df_cpi = pd.read_csv('../data/cleaned_cpi.csv')
df_cpi['date'] = pd.to_datetime(df_cpi['date'])
df_mpc = pd.read_csv('../data/processed_cpi_mpc.csv')
df_mpc['date'] = pd.to_datetime(df_mpc['date'])

os.makedirs('../outputs', exist_ok=True)

## 2. Core Inflation Heatmap (Year x Month)
This heatmap helps identify long-running trends and inflation episodes.

In [ ]:
df_cpi['year'] = df_cpi['date'].dt.year
df_cpi['month'] = df_cpi['date'].dt.month

# Pivot for heatmap (core_yoy)
heat = df_cpi.pivot_table(index='year', columns='month', values='cpi_core_yoy', aggfunc='mean')

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(heat, cmap='RdYlGn_r', center=4, annot=True, fmt='.1f',
            linewidths=0.3, ax=ax, cbar_kws={'label': 'Core CPI YoY %'})
ax.set_title('India Core CPI Inflation (%) — Monthly, 2015–2025\nGreen = Below 4% target band | Red = Elevated', fontsize=12, pad=10)
ax.set_xlabel('Month')
ax.set_ylabel('Year')
plt.tight_layout()
plt.savefig('../outputs/01_core_cpi_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. CPI Components vs RBI Repo Rate
We plot headline, core, food, and fuel inflation alongside the repo rate, highlighting MPC rate hikes and cuts.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# Top Panel: Component lines
ax1 = axes[0]
ax1.plot(df_cpi['date'], df_cpi['cpi_general_yoy'], color='#2c3e50', linewidth=2.5, label='Headline CPI')
ax1.plot(df_cpi['date'], df_cpi['cpi_core_yoy'], color='#e74c3c', linewidth=2, linestyle='--', label='Core (demand-side)')
ax1.plot(df_cpi['date'], df_cpi['cpi_food_yoy'], color='#27ae60', linewidth=1.5, alpha=0.8, label='Food (supply-side)')
ax1.plot(df_cpi['date'], df_cpi['cpi_fuel_yoy'], color='#f39c12', linewidth=1.5, alpha=0.8, label='Fuel')
ax1.axhline(y=4, color='grey', linestyle=':', linewidth=1, label='RBI 4% target')
ax1.axhline(y=6, color='red', linestyle=':', linewidth=1, alpha=0.5, label='Upper tolerance band')
ax1.legend(fontsize=9)
ax1.set_ylabel('YoY Inflation %')
ax1.set_title('India CPI Components vs. RBI Monetary Policy Repo Rate')
ax1.grid(True, alpha=0.2)

# Bottom Panel: Repo rate
ax2 = axes[1]
# Map repo rate to cpi dates
df_cpi_sorted = df_cpi.sort_values('date')
df_mpc_sorted = df_mpc.sort_values('date')
df_cpi_sorted['repo_rate'] = df_cpi_sorted['date'].map(dict(zip(df_mpc_sorted['date'], df_mpc_sorted['repo_rate']))).ffill()

ax2.step(df_cpi_sorted['date'], df_cpi_sorted['repo_rate'], color='#8e44ad', linewidth=2, label='Repo Rate %')

# Scatter hikes and cuts
hikes = df_mpc[df_mpc['decision'] == 'hike']
cuts = df_mpc[df_mpc['decision'] == 'cut']
ax2.scatter(hikes['date'], hikes['repo_rate'], color='red', zorder=5, s=60, label='Rate Hike', marker='^')
ax2.scatter(cuts['date'], cuts['repo_rate'], color='green', zorder=5, s=60, label='Rate Cut', marker='v')
ax2.set_ylabel('Repo Rate %')
ax2.set_xlabel('Date')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('../outputs/02_cpi_vs_repo_rate.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Variance Decomposition and Correlations
We check which components correlate most strongly with headline CPI, and save the annual summary table.

In [ ]:
components = {
    'Food (supply)': df_cpi['cpi_food_yoy'],
    'Fuel (supply)': df_cpi['cpi_fuel_yoy'],
    'Core (demand)': df_cpi['cpi_core_yoy']
}

print("Correlation of each component with Headline CPI:")
for name, series in components.items():
    valid = df_cpi[['cpi_general_yoy']].join(series.rename('comp')).dropna()
    corr = valid['cpi_general_yoy'].corr(valid['comp'])
    print(f"  {name}: {corr:.3f}")

# Year-wise averages
yearly = df_cpi.groupby('year')[['cpi_general_yoy', 'cpi_food_yoy', 'cpi_fuel_yoy', 'cpi_core_yoy']].mean().round(2)
print("\nYear-wise average inflation by component:")
print(yearly)

# Save CSV
yearly.to_csv('../outputs/yearly_cpi_components.csv')
print("\nYearly averages saved to outputs/yearly_cpi_components.csv")